In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType
)

from datetime import datetime, timezone
import uuid

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

TABLES = {
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "products": f"{CATALOG}.{SCHEMA}.silver_products",
    "orders": f"{CATALOG}.{SCHEMA}.silver_orders",
    "order_items": f"{CATALOG}.{SCHEMA}.silver_order_items",
    "payments": f"{CATALOG}.{SCHEMA}.silver_payments",
    "shipments": f"{CATALOG}.{SCHEMA}.silver_shipments",
    "inventory": f"{CATALOG}.{SCHEMA}.silver_inventory"
}

REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.business_rule_validation_report"
)

RUN_ID = str(uuid.uuid4())

CHECKED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

print("=" * 75)
print("BUSINESS RULE VALIDATION CONFIGURATION")
print("=" * 75)

print("Run ID:", RUN_ID)
print("Report table:", REPORT_TABLE)
print("Validation timestamp:", CHECKED_AT)


## 3. Load Silver tables

dfs = {}

for dataset_name, table_name in TABLES.items():

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Silver table does not exist: {table_name}"
        )

    dfs[dataset_name] = spark.table(table_name)

    print(
        f"{dataset_name}: "
        f"{dfs[dataset_name].count()} records loaded"
    )

customers = dfs["customers"]
products = dfs["products"]
orders = dfs["orders"]
order_items = dfs["order_items"]
payments = dfs["payments"]
shipments = dfs["shipments"]
inventory = dfs["inventory"]


## 4. Helper functions

results = []


def get_column(df, candidates):

    available = {
        column.lower(): column
        for column in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in available:
            return available[candidate.lower()]

    return None


def add_result(
    dataset_name,
    rule_name,
    description,
    total_records,
    failed_records,
    severity="ERROR"
):

    total_records = int(total_records or 0)
    failed_records = int(failed_records or 0)

    if severity == "INFO":

        status = "INFO"

    elif failed_records == 0:

        status = "PASS"

    elif severity == "WARNING":

        status = "WARN"

    else:

        status = "FAIL"

    failure_percentage = (
        round(
            (failed_records / total_records) * 100,
            2
        )
        if total_records > 0
        else 0.0
    )

    results.append({

        "checked_at": CHECKED_AT,

        "dataset_name": dataset_name,

        "description": description,

        "failed_records": failed_records,

        "failure_percentage": float(failure_percentage),

        "rule_name": rule_name,

        "severity": severity,

        "status": status,

        "total_records": total_records

    })


def add_info(
    dataset_name,
    rule_name,
    description
):

    add_result(
        dataset_name,
        rule_name,
        description,
        0,
        0,
        "INFO"
    )


def orphan_count(
    child_df,
    child_key,
    parent_df,
    parent_key
):

    child = (
        child_df
        .filter(F.col(child_key).isNotNull())
        .alias("child")
    )

    parent = (
        parent_df
        .select(
            F.col(parent_key).alias("_parent_key")
        )
        .filter(F.col("_parent_key").isNotNull())
        .dropDuplicates()
        .alias("parent")
    )

    return (
        child
        .join(
            parent,
            F.col(f"child.{child_key}") ==
            F.col("parent._parent_key"),
            "left_anti"
        )
        .count()
    )


def safe_count(df, condition):

    return df.filter(condition).count()


def count_distinct_non_null(df, column_name):

    return (
        df
        .filter(F.col(column_name).isNotNull())
        .select(column_name)
        .distinct()
        .count()
    )



## 5. Referential integrity validation

### 5.1 Orders without customers

total = orders.count()

failed = orphan_count(
    orders,
    "customer_id",
    customers,
    "customer_id"
)

add_result(
    "silver_orders",
    "order_without_customer",
    "Every non-null order customer_id must exist in Silver customers.",
    total,
    failed
)

print("order_without_customer:", failed)


### 5.2 Order items without orders

total = order_items.count()

failed = orphan_count(
    order_items,
    "order_id",
    orders,
    "order_id"
)

add_result(
    "silver_order_items",
    "order_item_without_order",
    "Every non-null order item order_id must exist in Silver orders.",
    total,
    failed
)

print("order_item_without_order:", failed)


### 5.3 Order items without products

total = order_items.count()

failed = orphan_count(
    order_items,
    "product_id",
    products,
    "product_id"
)

add_result(
    "silver_order_items",
    "order_item_without_product",
    "Every non-null order item product_id must exist in Silver products.",
    total,
    failed
)

print("order_item_without_product:", failed)


### 5.4 Payments without orders

total = payments.count()

failed = orphan_count(
    payments,
    "order_id",
    orders,
    "order_id"
)

add_result(
    "silver_payments",
    "payment_without_order",
    "Every non-null payment order_id must exist in Silver orders.",
    total,
    failed
)

print("payment_without_order:", failed)


### 5.5 Shipments without orders

total = shipments.count()

failed = orphan_count(
    shipments,
    "order_id",
    orders,
    "order_id"
)

add_result(
    "silver_shipments",
    "shipment_without_order",
    "Every non-null shipment order_id must exist in Silver orders.",
    total,
    failed
)

print("shipment_without_order:", failed)


### 5.6 Inventory without products

total = inventory.count()

failed = orphan_count(
    inventory,
    "product_id",
    products,
    "product_id"
)

add_result(
    "silver_inventory",
    "inventory_without_product",
    "Every non-null inventory product_id must exist in Silver products.",
    total,
    failed
)

print("inventory_without_product:", failed)


### 5.7 Orders without items

total = orders.count()

order_status_col = get_column(
    orders,
    ["order_status"]
)

order_item_order_ids = (
    order_items
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .dropDuplicates()
)

orders_without_items_df = (
    orders
    .filter(F.col("order_id").isNotNull())
    .join(
        order_item_order_ids,
        "order_id",
        "left_anti"
    )
)

if order_status_col:

    orders_without_items_df = (
        orders_without_items_df
        .filter(
            ~F.lower(
                F.trim(
                    F.coalesce(
                        F.col(order_status_col).cast("string"),
                        F.lit("")
                    )
                )
            ).isin("cancelled", "canceled")
        )
    )

failed = orders_without_items_df.count()

add_result(
    "silver_orders",
    "orders_without_items",
    "Non-cancelled orders should have at least one associated order item.",
    total,
    failed
)

print("orders_without_items:", failed)


## 6. Inventory business rule validation

### 6.1 Duplicate inventory business key

total = inventory.count()

inventory_business_key = [
    "product_id",
    "snapshot_date",
    "warehouse_id"
]

missing_key_condition = (
    F.col("product_id").isNull() |
    F.col("snapshot_date").isNull() |
    F.col("warehouse_id").isNull()
)

inventory_valid_keys = inventory.filter(
    ~missing_key_condition
)

inventory_duplicate_keys = (
    inventory_valid_keys
    .groupBy(*inventory_business_key)
    .count()
    .filter(F.col("count") > 1)
)

# Count excess duplicate rows, not just duplicate combinations.

failed = (
    inventory_duplicate_keys
    .agg(
        F.sum(F.col("count") - F.lit(1)).alias("duplicates")
    )
    .first()["duplicates"]
    or 0
)

add_result(
    "silver_inventory",
    "duplicate_inventory_business_key",
    "The combination of product_id, snapshot_date and warehouse_id must be unique.",
    total,
    failed
)

print("duplicate_inventory_business_key:", failed)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 6.2 Invalid inventory quantity

total = inventory.count()

quantity_col = get_column(
    inventory,
    [
        "quantity_on_hand",
        "quantity",
        "stock_quantity",
        "available_quantity",
        "inventory_quantity"
    ]
)

if quantity_col:

    failed = safe_count(
        inventory,
        F.col(quantity_col).isNotNull() &
        (
            F.col(quantity_col).cast("double") < 0
        )
    )

    add_result(
        "silver_inventory",
        "invalid_inventory_quantity",
        "Inventory quantity must not be negative.",
        total,
        failed
    )

else:

    add_info(
        "silver_inventory",
        "invalid_inventory_quantity",
        "Skipped: inventory quantity column was not found."
    )

print("invalid_inventory_quantity:", failed if quantity_col else "SKIPPED")


## 7. Product pricing validation

### 7.1 Selling price below product cost

total = products.count()

selling_price_col = get_column(
    products,
    ["unit_price", "selling_price", "price", "sale_price"]
)

cost_col = get_column(
    products,
    ["unit_cost", "cost_price", "product_cost", "cost"]
)

if selling_price_col and cost_col:

    failed = safe_count(
        products,
        F.col(selling_price_col).isNotNull() &
        F.col(cost_col).isNotNull() &
        (
            F.col(selling_price_col).cast("double") <
            F.col(cost_col).cast("double")
        )
    )

    add_result(
        "silver_products",
        "selling_price_below_product_cost",
        "Product selling price should not be lower than product cost.",
        total,
        failed,
        "WARNING"
    )

else:

    add_info(
        "silver_products",
        "selling_price_below_product_cost",
        "Skipped: selling price or product cost column was not found."
    )

print(
    "selling_price_below_product_cost:",
    failed if selling_price_col and cost_col else "SKIPPED"
)


## 8. Shipment date validation

### 8.1 Shipment date consistency

total = shipments.count()

shipped_col = get_column(
    shipments,
    ["shipped_date", "ship_date", "shipped_at"]
)

delivered_col = get_column(
    shipments,
    ["delivered_date", "delivery_date", "delivered_at"]
)

if shipped_col and delivered_col:

    failed = safe_count(
        shipments,
        F.col(shipped_col).isNotNull() &
        F.col(delivered_col).isNotNull() &
        (
            F.to_timestamp(F.col(delivered_col)) <
            F.to_timestamp(F.col(shipped_col))
        )
    )

    add_result(
        "silver_shipments",
        "shipment_date_consistency",
        "Delivered date must not be earlier than shipped date.",
        total,
        failed
    )

else:

    add_info(
        "silver_shipments",
        "shipment_date_consistency",
        "Skipped: shipment or delivery date column was not found."
    )

print(
    "shipment_date_consistency:",
    failed if shipped_col and delivered_col else "SKIPPED"
)

## 9. Payment business rule validation

payment_status_col = get_column(
    payments,
    ["payment_status", "status", "transaction_status"]
)

if payment_status_col:

    successful_payments = payments.filter(
        F.lower(
            F.trim(
                F.col(payment_status_col).cast("string")
            )
        ).isin(
            "success",
            "successful",
            "paid",
            "completed"
        )
    )

    successful_order_ids = (
        successful_payments
        .filter(F.col("order_id").isNotNull())
        .select("order_id")
        .dropDuplicates()
    )


    # 9.1 Orders without successful payment
    

    total = orders.count()

    failed = (
        orders
        .filter(F.col("order_id").isNotNull())
        .join(
            successful_order_ids,
            "order_id",
            "left_anti"
        )
        .count()
    )

    add_result(
        "silver_orders",
        "orders_without_successful_payment",
        "Orders without successful payments require review. Pending and unpaid orders may be legitimate.",
        total,
        failed,
        "WARNING"
    )

    print("orders_without_successful_payment:", failed)


    # 9.2 Multiple successful payments

    successful_payment_counts = (
        successful_payments
        .filter(F.col("order_id").isNotNull())
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
    )

    failed = successful_payment_counts.count()

    add_result(
        "silver_payments",
        "multiple_successful_payments",
        "Orders with multiple successful payment records require review.",
        successful_payments.count(),
        failed,
        "INFO"
    )

    print("multiple_successful_payments:", failed)


    # 9.3 Successful payment exceeds order total

    order_total_col = get_column(
        orders,
        ["order_total", "total_amount", "total_price", "grand_total"]
    )

    payment_amount_col = get_column(
        payments,
        ["amount", "payment_amount", "paid_amount", "transaction_amount"]
    )

    if order_total_col and payment_amount_col:

        successful_payment_totals = (
            successful_payments
            .filter(F.col("order_id").isNotNull())
            .groupBy("order_id")
            .agg(
                F.sum(
                    F.col(payment_amount_col).cast("double")
                ).alias("_successful_payment_total")
            )
        )

        order_amounts = orders.select(
            "order_id",
            F.col(order_total_col)
            .cast("double")
            .alias("_order_total")
        )

        payment_comparison = (
            successful_payment_totals
            .join(
                order_amounts,
                "order_id",
                "inner"
            )
        )

        failed = safe_count(
            payment_comparison,
            F.col("_successful_payment_total").isNotNull() &
            F.col("_order_total").isNotNull() &
            (
                F.col("_successful_payment_total") >
                F.col("_order_total") + F.lit(0.01)
            )
        )

        add_result(
            "silver_payments",
            "successful_payment_exceeds_order_total",
            "Total successful payments must not exceed the associated order total.",
            payment_comparison.count(),
            failed
        )

        print("successful_payment_exceeds_order_total:", failed)

    else:

        add_info(
            "silver_payments",
            "successful_payment_exceeds_order_total",
            "Skipped: payment amount or order total column was not found."
        )

else:

    add_info(
        "silver_orders",
        "orders_without_successful_payment",
        "Skipped: payment status column was not found."
    )

    add_info(
        "silver_payments",
        "multiple_successful_payments",
        "Skipped: payment status column was not found."
    )

    add_info(
        "silver_payments",
        "successful_payment_exceeds_order_total",
        "Skipped: payment status column was not found."
    )


## 10. Order total reconciliation

order_total_col = get_column(
    orders,
    ["order_total", "total_amount", "total_price", "grand_total"]
)

line_total_col = get_column(
    order_items,
    ["line_total"]
)

if order_total_col and line_total_col:

    item_line_totals = (
        order_items
        .filter(F.col("order_id").isNotNull())
        .groupBy("order_id")
        .agg(
            F.sum(
                F.col(line_total_col).cast("double")
            ).alias("_calculated_line_total")
        )
    )

    order_totals = (
        orders
        .filter(F.col("order_id").isNotNull())
        .select(
            "order_id",
            F.col(order_total_col)
            .cast("double")
            .alias("_stored_order_total")
        )
    )

    reconciliation = (
        order_totals
        .join(
            item_line_totals,
            "order_id",
            "inner"
        )
    )

    failed = safe_count(
        reconciliation,
        F.col("_stored_order_total").isNotNull() &
        F.col("_calculated_line_total").isNotNull() &
        (
            F.abs(
                F.col("_stored_order_total") -
                F.col("_calculated_line_total")
            ) > F.lit(0.01)
        )
    )

    add_result(
        "silver_orders",
        "order_total_reconciliation",
        "Stored order_total should match the sum of order_items.line_total within 0.01 tolerance.",
        reconciliation.count(),
        failed
    )

    print("order_total_reconciliation:", failed)

else:

    add_info(
        "silver_orders",
        "order_total_reconciliation",
        "Skipped: order_total or line_total column was not found."
    )


## 11. Validate all business rule results

print("=" * 75)
print("BUSINESS RULE VALIDATION RESULTS")
print("=" * 75)

print("Total rules:", len(results))

for result in results:

    print(
        result["rule_name"],
        "|",
        result["status"],
        "| Failed records:",
        result["failed_records"],
        "| Severity:",
        result["severity"]
    )


## 12. Create report DataFrame

report_schema = StructType([

    StructField(
        "checked_at",
        TimestampType(),
        True
    ),

    StructField(
        "dataset_name",
        StringType(),
        True
    ),

    StructField(
        "description",
        StringType(),
        True
    ),

    StructField(
        "failed_records",
        LongType(),
        True
    ),

    StructField(
        "failure_percentage",
        DoubleType(),
        True
    ),

    StructField(
        "rule_name",
        StringType(),
        True
    ),

    StructField(
        "severity",
        StringType(),
        True
    ),

    StructField(
        "status",
        StringType(),
        True
    ),

    StructField(
        "total_records",
        LongType(),
        True
    )

])

report_df = spark.createDataFrame(
    results,
    schema=report_schema
)

print("Report DataFrame created.")

display(
    report_df.orderBy(
        "dataset_name",
        "rule_name"
    )
)


## 13. Validate report before saving

if len(results) != 15:

    raise ValueError(
        f"Expected 15 business rules, but generated {len(results)}."
    )

invalid_statuses = report_df.filter(
    ~F.col("status").isin(
        "PASS",
        "FAIL",
        "WARN",
        "INFO"
    )
).count()

if invalid_statuses > 0:

    raise ValueError(
        f"Report contains {invalid_statuses} invalid status records."
    )

duplicate_rule_names = (
    report_df
    .groupBy("rule_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_rule_names > 0:

    raise ValueError(
        "Duplicate rule names found in the current validation run."
    )

print("Report validation passed.")


## 14. Save business rule validation report

if not spark.catalog.tableExists(REPORT_TABLE):

    (
        report_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(REPORT_TABLE)
    )

else:

    (
        report_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(REPORT_TABLE)
    )

print("Business rule report saved successfully:")
print(REPORT_TABLE)


## 15. Final validation summary

saved_report = spark.table(REPORT_TABLE)

current_run_report = saved_report.filter(
    F.col("checked_at") == F.lit(CHECKED_AT)
)

total_rules = current_run_report.count()

passed_rules = current_run_report.filter(
    F.col("status") == "PASS"
).count()

failed_rules = current_run_report.filter(
    F.col("status") == "FAIL"
).count()

warning_rules = current_run_report.filter(
    F.col("status") == "WARN"
).count()

info_rules = current_run_report.filter(
    F.col("status") == "INFO"
).count()

if total_rules != 15:

    raise RuntimeError(
        f"Expected 15 saved rules, found {total_rules}."
    )

print("=" * 75)
print("BUSINESS RULE VALIDATION COMPLETED")
print("=" * 75)

print("Run ID              :", RUN_ID)
print("Report table        :", REPORT_TABLE)
print("Total rules         :", total_rules)
print("Passed rules        :", passed_rules)
print("Failed rules        :", failed_rules)
print("Warning rules       :", warning_rules)
print("Info / skipped rules:", info_rules)

print("-" * 75)

print(
    "Pass percentage:",
    round(
        (passed_rules / total_rules) * 100,
        2
    ),
    "%"
)

print(
    "Historical records:",
    saved_report.count()
)

print("=" * 75)

display(
    current_run_report.orderBy(
        "dataset_name",
        "rule_name"
    )
)


## 16. Verify final report table

spark.sql(
    f"DESCRIBE TABLE {REPORT_TABLE}"
).show(truncate=False)

print("Final report record count:", saved_report.count())


## 17. Business rule failure breakdown

display(
    current_run_report
    .groupBy(
        "dataset_name",
        "status",
        "severity"
    )
    .agg(
        F.count("*").alias("total_rules"),
        F.sum("failed_records").alias("total_failed_records")
    )
    .orderBy(
        "dataset_name",
        "status"
    )
)


In [0]:
# 1. Orders without order items

print("=" * 70)
print("ORDERS WITHOUT ORDER ITEMS")
print("=" * 70)

orders_without_items = (
    orders
    .join(
        order_items
        .select("order_id")
        .filter(F.col("order_id").isNotNull())
        .dropDuplicates(),
        "order_id",
        "left_anti"
    )
)

print("Missing order count:", orders_without_items.count())

display(orders_without_items.limit(20))

# COMMAND ----------

# 2. Duplicate inventory business keys

print("=" * 70)
print("DUPLICATE INVENTORY BUSINESS KEYS")
print("=" * 70)

print("Detected snapshot column:", snapshot_col)

if snapshot_col:
    duplicate_inventory_keys = (
        inventory
        .filter(
            F.col("product_id").isNotNull() &
            F.col(snapshot_col).isNotNull()
        )
        .groupBy("product_id", snapshot_col)
        .agg(
            F.count("*").alias("record_count")
        )
        .filter(F.col("record_count") > 1)
        .orderBy(F.desc("record_count"))
    )

    print(
        "Duplicate key combinations:",
        duplicate_inventory_keys.count()
    )

    display(duplicate_inventory_keys.limit(30))

# COMMAND ----------

# 3. Order total reconciliation

print("=" * 70)
print("ORDER TOTAL RECONCILIATION")
print("=" * 70)

print("Order total column:", order_total_col)
print("Item quantity column:", quantity_col)
print("Item price column:", item_price_col)

if order_total_col and quantity_col and item_price_col:

    item_totals_diagnostic = (
        order_items
        .filter(F.col("order_id").isNotNull())
        .groupBy("order_id")
        .agg(
            F.sum(
                F.col(quantity_col).cast("double") *
                F.col(item_price_col).cast("double")
            ).alias("calculated_item_total")
        )
    )

    order_totals_diagnostic = orders.select(
        "order_id",
        F.col(order_total_col)
        .cast("double")
        .alias("stored_order_total")
    )

    reconciliation_diagnostic = (
        order_totals_diagnostic
        .join(item_totals_diagnostic, "order_id", "inner")
        .withColumn(
            "difference",
            F.round(
                F.col("stored_order_total") -
                F.col("calculated_item_total"),
                2
            )
        )
        .withColumn(
            "absolute_difference",
            F.abs(F.col("difference"))
        )
        .orderBy(F.desc("absolute_difference"))
    )

    display(reconciliation_diagnostic.limit(30))

    print(
        "Compared orders:",
        reconciliation_diagnostic.count()
    )

    print(
        "Orders with a difference:",
        reconciliation_diagnostic.filter(
            F.col("absolute_difference") > 0.01
        ).count()
    )

# COMMAND ----------

# 4. Inspect available columns

print("=" * 70)
print("ACTUAL SILVER TABLE COLUMNS")
print("=" * 70)

for name, df in dfs.items():
    print(f"\n{name}:")
    print(df.columns)
